In [1]:
import math
import random
import numpy as np
import time
from waveNetArchitecture import Value, Linear, BatchNorm1D, LayerNorm, Tanh, ReLU, Embedding, FlattenConsecutive, Sequential, cross_entropy, BagOfWords, PositionalEmbedding, Head, MultiHead, FeedForward, Block, AdamW
from bpeTokenizer import RegexTokenizer
with open('fineweb_edu_subset.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
tok = RegexTokenizer.load('shakespearTokenizer.json')
vocab_size = tok.vocab_size
encode = tok.encode
decode = tok.decode

blockSize = 64 # Increase the amount of context
inputs, outputs = [],[]
print(vocab_size)

1024


In [3]:
data = np.array(encode(text)) # plain int array: this is data, not a parameter, so it needs no autograd
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]
print(data[:20], '| train:', len(train_data), 'val:', len(val_data))

[354 292 266 509 468 338 972 300 101  10 480 395 267 519  44  32 423 645
 297 855] | train: 3309437 val: 367716


In [4]:
n_embd = 32     # embedding dimensionality (must divide evenly across num_heads)
num_heads = 4   # attention heads per block
n_blocks = 4    # how many transformer blocks to stack

model = Sequential([
  Embedding(vocab_size, n_embd),
  PositionalEmbedding(blockSize, n_embd),
  *[Block(n_embd, num_heads, blockSize) for _ in range(n_blocks)],
  LayerNorm(n_embd),
  Linear(n_embd, vocab_size),
])

# keep initial predictions close to uniform, so the starting loss is near -log(1/vocab_size)
model.layers[-1].weight.data *= 0.1

parameters = model.parameters()
print(sum(p.data.size for p in parameters)) # number of parameters in total

119104


In [7]:
max_steps = 30000   # 200k is unreachable on numpy/CPU; raise it once you see the curve still falling
batch_size = 32
eval_every = 1000   # how often to measure held-out loss
eval_iters = 10     # batches averaged per measurement (more = less noisy, but costs forward passes)

lossi = []      # raw per-step training loss. NOT log10 - the plot cell takes the log if it wants one
val_lossi = []  # (step, train_estimate, val_estimate), recorded every eval_every steps
ud = []

opt = AdamW(parameters, lr=1e-3, betas=(0.9,0.95), weight_decay=0.01)

# build sliding-window dataset: each input is `blockSize` consecutive tokens,
# targets are that same window shifted one to the right - a prediction at every position,
# not just one at the end (that's the whole point of keeping T alive through attention)
def make_windows(tokens, blockSize):
    # sliding_window_view returns a *view*, so this is free. The old list-of-arrays
    # version materialized every window: ~560MB for Shakespeare.
    n = len(tokens) - blockSize
    X = np.lib.stride_tricks.sliding_window_view(tokens, blockSize)[:n]      # (n, blockSize) context
    Y = np.lib.stride_tricks.sliding_window_view(tokens[1:], blockSize)[:n]  # (n, blockSize) same, shifted by one
    return X, Y

# built from train_data and val_data separately, so no val window is ever trained on
Xtr, Ytr = make_windows(train_data, blockSize)
Xva, Yva = make_windows(val_data, blockSize)
print(f'train windows: {Xtr.shape[0]:,}   val windows: {Xva.shape[0]:,}')

def split_loss(X, Y, iters=eval_iters):
    """Mean loss over `iters` random batches. Forward pass only - we never call
    .backward() here, so each graph is discarded instead of accumulating grads."""
    losses = []
    for _ in range(iters):
        ix = np.random.randint(0, X.shape[0], (batch_size,))
        losses.append(float(cross_entropy(model(X[ix]), Y[ix]).data))
    return np.mean(losses)

for i in range(max_steps):

    # minibatch construct - drawn from the TRAIN windows only
    ix = np.random.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]  # (batch_size, blockSize), (batch_size, blockSize)

    # forward pass
    logits = model(Xb)               # (batch_size, blockSize, vocab_size)
    loss = cross_entropy(logits, Yb) # loss function

    # backward pass
    # for p in parameters:
    #     p.grad = np.zeros_like(p.data, dtype=float)
    opt.zero_grad()
    loss.backward()

    lr = 1e-3 if i < 0.8 * max_steps else 1e-4
    opt.step(lr)
    # for p in parameters:
    #     p.data += -lr * p.grad

    # track stats
    lossi.append(float(loss.data))
    ud.append(opt.ud)

    if i % 30 == 0: # cheap running average of the last 200 batches, no extra compute
        print(f'{i:7d}/{max_steps:7d}: {np.mean(lossi[-200:]):.4f}')
    if i % eval_every == 0 or i == max_steps - 1:
        tr, va = split_loss(Xtr, Ytr), split_loss(Xva, Yva)
        val_lossi.append((i, tr, va))
        print(f'{i:7d}/{max_steps:7d}: train {tr:.4f}  val {va:.4f}  (gap {va-tr:+.4f})')

train windows: 3,309,373   val windows: 367,652
      0/  30000: 6.9264
      0/  30000: train 6.9016  val 6.9032  (gap +0.0017)
     30/  30000: 6.4925
     60/  30000: 6.1977
     90/  30000: 6.0441
    120/  30000: 5.9609
    150/  30000: 5.8992
    180/  30000: 5.8485
    210/  30000: 5.7444
    240/  30000: 5.6234
    270/  30000: 5.5514
    300/  30000: 5.4820
    330/  30000: 5.4056
    360/  30000: 5.3283
    390/  30000: 5.2558
    420/  30000: 5.1870
    450/  30000: 5.1250
    480/  30000: 5.0702
    510/  30000: 5.0205
    540/  30000: 4.9781
    570/  30000: 4.9359
    600/  30000: 4.8968
    630/  30000: 4.8615
    660/  30000: 4.8284
    690/  30000: 4.7950
    720/  30000: 4.7619
    750/  30000: 4.7337
    780/  30000: 4.7074
    810/  30000: 4.6853
    840/  30000: 4.6651
    870/  30000: 4.6456
    900/  30000: 4.6229
    930/  30000: 4.6069
    960/  30000: 4.5898
    990/  30000: 4.5724
   1000/  30000: train 4.5374  val 4.5284  (gap -0.0089)
   1020/  30000: 4.556

KeyboardInterrupt: 

In [ ]:
# sample from the model
rng = np.random

out = []
context = [0] * blockSize
while len(out) < 10000:
    x = np.array([context]) # (1, blockSize); the Embedding layer looks these up
    logits = model(x)                     # (1, blockSize, vocab_size): a prediction at every position
    last_logits = logits.data[:, -1, :]    # only the newest position matters for generation
    e = np.exp(last_logits - last_logits.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)
    # sample from the distribution
    ix = rng.choice(vocab_size, p=probs[0])
    # shift the context window and track the samples
    context = context[1:] + [ix]
    out.append(ix)


print(tok.decode(out))

 is me Unfectlymally pren'tBe vegcking settude of this spite in which/ckerallellows plan’dature15iegine Centuring sergy anturehround of Biphritutifichemoric Cul Wata's the funcienteratsignation and stainly bear as hist to lost ske local presefovey it.

Tems Can Ylay of humber Researto ba’ser unle is made to failive the nyry back in access tlease of some young to one nrants of acides billing, will peorbruited curge enges.
- Ate. Ogan. The hride that areas halicon. Wolude about Del Als; Scervare. PasedR. Howeder to sea covernettime
- Emericanso. The Eganc, (Hustistsite Joumenturris- Di betw good 2 1198. The Wigners Caps, Soman that Millier tad. Wals their most forestings there isso to laugues of relvered forwitters, the normentrof. Onso; and jud confutlyduged the Isoan pure fiitientrature in sevents.
For expeo both it effortment plamplexiious your counters then use. The Active C-Bamins, which of anything programal Flinumpton yearch Ce) and CI on following Raf
er. Rithinal eminging lays 

In [11]:
# encode a query and use it as the seed context, instead of starting from all padding
rng = np.random

query = input("Query: ")
query_tokens = encode(query)

# left-pad with the pad token (0) if the prompt is shorter than blockSize,
# or keep only the most recent blockSize tokens if it's longer
if len(query_tokens) < blockSize:
    context = [0] * (blockSize - len(query_tokens)) + query_tokens
else:
    context = query_tokens[-blockSize:]

print("Thinking...")

num_new_tokens = 300
out = []  # only the newly generated tokens - the query itself gets cut from the printed response

for _ in range(num_new_tokens):
    x = np.array([context])              # (1, blockSize); the Embedding layer looks these up
    logits = model(x)                    # (1, blockSize, vocab_size): a prediction at every position
    last_logits = logits.data[:, -1, :]  # only the newest position matters for generation
    e = np.exp(last_logits - last_logits.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)
    # sample from the distribution
    ix = rng.choice(vocab_size, p=probs[0])
    # shift the context window and track the samples
    context = context[1:] + [ix]
    out.append(ix)

print(f"\nQuery: {query}\nResponse: {tok.decode(out)}")

Thinking...

Query: Solve for x in 5x = 24
Response: 963, an capio mean ob- Auryearchises, Propes were them in Birright is bircing that so wit sucted in propultralls
-and”
Ded Jellely partice, hitive providduding performationing plient of women no deal higain; reques not world, diffiled more than only sort of the differentry forcurescies how the percent of it of revigatated by comportagese that are a saderver like peoplement of summand the win Bears in most termoms like they spathering diseart is nohyda from And:
BDIN Treloredved oil down. NOI carm Masesing a particiaf Tellineencolitorors is securreems from the potaged to pittle may be adased to� the most sever,
-The lech,
" Hail Howugan. This can be a neUnian d providable to make the farchole


In [22]:
# Basic attention bit

B, T, C = 4, 8, 2 
x = np.random.randn(B, T, C)
print(x.shape)

(4, 8, 2)


In [23]:
xbow = np.zeros((B,T,C)) # Bag of Words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = np.mean(xprev, 0)

xbow[2]

array([[-0.83360006,  0.45040839],
       [-1.10059142, -0.4697657 ],
       [-1.03429883, -1.12052612],
       [-1.01487549, -0.6256353 ],
       [-0.60080551, -0.47944088],
       [-0.61257341, -0.46789691],
       [-0.7301151 , -0.64742479],
       [-0.64868054, -0.55582508]])

In [24]:
wei = np.tril(np.ones((T, T)))
wei = wei / wei.sum(1, keepdims=True)
xbow2 = wei @ x
xbow2[2]

array([[-0.83360006,  0.45040839],
       [-1.10059142, -0.4697657 ],
       [-1.03429883, -1.12052612],
       [-1.01487549, -0.6256353 ],
       [-0.60080551, -0.47944088],
       [-0.61257341, -0.46789691],
       [-0.7301151 , -0.64742479],
       [-0.64868054, -0.55582508]])

In [7]:
# Self attention
B,T,C = 4,8,32 # Batch, Time, Channels
x = Value(np.random.randn(B, T, C))

head = Head(C, head_size=16, block_size=T)
out = head(x)
out.shape

(4, 8, 16)

In [ ]:
from exportGPTWeights import export_gpt_weights
export_gpt_weights(model, {i: b.decode('utf-8', errors='replace') for i, b in tok.vocab.items()})

Wrote docs/gptWeights.json (584.2 KB) - 56,769 params, vocab 65, n_embd 32, 4 blocks x 4 heads, block size 64


'docs/gptWeights.json'